In [9]:
import pandas as pd
import re
import torch
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames:
        print("   ", filename)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/nikhilsharmaai
/kaggle/input/datasets/nikhilsharmaai/text-summarizer-data-set
    samsum-train.csv
    samsum-test.csv
    samsum-validation.csv


In [2]:
import pandas as pd

DATA_PATH = "/kaggle/input/datasets/nikhilsharmaai/text-summarizer-data-set"

train_data = pd.read_csv(f"{DATA_PATH}/samsum-train.csv")
val_data = pd.read_csv(f"{DATA_PATH}/samsum-validation.csv")
test_data = pd.read_csv(f"{DATA_PATH}/samsum-test.csv")

In [3]:
print(train_data.head())
print(train_data.shape)

print(val_data.head())
print(val_data.shape)

print(test_data.head())
print(test_data.shape)

         id                                           dialogue  \
0  13818513  Amanda: I baked  cookies. Do you want some?\r\...   
1  13728867  Olivia: Who are you voting for in this electio...   
2  13681000  Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...   
3  13730747  Edward: Rachel, I think I'm in ove with Bella....   
4  13728094  Sam: hey  overheard rick say something\r\nSam:...   

                                             summary  
0  Amanda baked cookies and will bring Jerry some...  
1  Olivia and Olivier are voting for liberals in ...  
2  Kim may try the pomodoro technique recommended...  
3  Edward thinks he is in love with Bella. Rachel...  
4  Sam is confused, because he overheard Rick com...  
(14732, 3)
         id                                           dialogue  \
0  13817023  A: Hi Tom, are you busy tomorrow’s afternoon?\...   
1  13716628  Emma: I’ve just fallen in love with this adven...   
2  13829420  Jackie: Madison is pregnant\r\nJackie: but she...  

In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

# Random Sampling

In [6]:
train_data = train_data.sample(n=6000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [11]:
# Preprocessing Text
def clean_data(text):
    text = re.sub(r"\r\n" ," ", text)
    text = re.sub(r"\rs+", " ", text)
    text = re.sub(r"<.*?>"," ", text)
    text = text.strip().lower()
    return text

In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenize

In [13]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
# raw data -> tokenzie inputs for fine tuning 

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [15]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [16]:
#input ids 
# 1 is the EOS , after that all 0s are padding 

# attention mask
# 1-> valid value 
# labels - target => summary

# Model

In [17]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [18]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: cuda


In [19]:
# Training arguments

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 10,
    weight_decay = 0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500,
    
)

In [20]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [21]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,0.772644
2,6.967389,0.713887
3,0.787946,0.699090
4,0.736190,0.690131
5,0.736190,0.684846
6,0.715013,0.681429
7,0.696643,0.680072
8,0.687011,0.679240
9,0.687011,0.678048
10,0.684399,0.677666


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=1.5480533040364584, metrics={'train_runtime': 1708.5995, 'train_samples_per_second': 35.116, 'train_steps_per_second': 2.195, 'total_flos': 8120508088320000.0, 'train_loss': 1.5480533040364584, 'epoch': 10.0})

In [22]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [23]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer= T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

****testing 

In [25]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)
    #tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )
    #generate the summary
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    ) 
    #token ids convert to summary
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary

In [24]:
import shutil

# Zip the folders
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working/results')
shutil.make_archive('/kaggle/working/saved_summary_model', 'zip', '/kaggle/working/saved_summary_model')

'/kaggle/working/saved_summary_model.zip'

In [26]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""
summary = summarize_dialogue(test_dialogue)
print("Summary: ",summary)

Summary:  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
